
# Synthetic Retail Data Warehouse Generator

This notebook generates a synthetic retail warehouse with:
- richer product titles and descriptions,
- more varied colors, sizes, materials, and attributes,
- dynamic review text with sentiment and detailed feedback,
- consistent foreign keys for analytics and RAG use cases.

Start with a smaller `SCALE` in Colab, then increase it when ready.


In [ ]:

# If needed in Colab, install dependencies.
!pip -q install pandas numpy


In [ ]:

import json
from pathlib import Path
from datetime import datetime, timedelta
import random
import numpy as np
import pandas as pd

# --------------------
# Parameters
# --------------------
OUT_DIR = Path("/content/retail_warehouse_out")
SCALE = 0.01   # set to 1.0 for the full dataset
SEED = 42

BASE_COUNTS = {
    "brands": 250,
    "categories": 180,
    "warehouses": 35,
    "products": 1_000_000,
    "product_variants": 2_500_000,
    "customers": 120_000,
    "customer_preferences": 120_000,
    "orders": 2_000_000,
    "order_items": 6_000_000,
    "order_tracking": 2_000_000,
    "reviews": 8_000_000,
    "inventory_snapshots": 5_000_000,
    "behavior_events": 500_000,
    "tool_call_logs": 500_000,
}

def scaled(n, s):
    return max(1, int(round(n * s)))

COUNTS = {k: scaled(v, SCALE) for k, v in BASE_COUNTS.items()}

rng = np.random.default_rng(SEED)
py_rng = random.Random(SEED)
OUT_DIR.mkdir(parents=True, exist_ok=True)

departments = [
    "Outdoor", "Clothing", "Electronics", "Home", "Fitness", "Kids", "Sports", "Beauty",
    "Kitchen", "Travel", "Pet", "Office", "Footwear", "Accessories"
]

product_templates = {
    "Outdoor": [
        "waterproof hiking jacket", "insulated shell jacket", "trail backpack", "packable rain shell",
        "fleece-lined softshell", "windproof trekking layer"
    ],
    "Clothing": [
        "everyday fleece hoodie", "all-weather jacket", "thermal base layer", "stretch jogger",
        "oversized knit sweater", "performance tee"
    ],
    "Electronics": [
        "wireless earbuds", "smartwatch", "portable speaker", "USB-C charger", "noise cancelling headset",
        "Bluetooth tracker"
    ],
    "Home": [
        "coffee maker", "blender", "air purifier", "table lamp", "storage basket", "vacuum accessory kit"
    ],
    "Fitness": [
        "yoga mat", "resistance band set", "foam roller", "adjustable dumbbell", "gym towel", "heart rate strap"
    ],
    "Kids": [
        "rain jacket", "school backpack", "lunch box", "water bottle", "play sneakers", "insulated snack bag"
    ],
    "Sports": [
        "running shoes", "training shorts", "sports cap", "compression socks", "match day backpack", "gym duffel"
    ],
    "Beauty": [
        "hydrating serum", "daily moisturizer", "sunscreen lotion", "hair mask", "cleansing balm", "face mist"
    ],
    "Kitchen": [
        "nonstick pan", "knife set", "meal prep container", "cutting board", "electric kettle", "baking tray"
    ],
    "Travel": [
        "carry-on backpack", "compression packing cubes", "travel pillow", "luggage tag set", "passport wallet", "toiletry kit"
    ],
    "Pet": [
        "pet bed", "leash set", "slow feeder bowl", "grooming brush", "raincoat", "travel water bowl"
    ],
    "Office": [
        "desk organizer", "ergonomic mouse", "notebook set", "monitor stand", "task lamp", "file holder"
    ],
    "Footwear": [
        "running shoes", "hiking boots", "slip-on sneakers", "training shoes", "winter boots", "walking shoes"
    ],
    "Accessories": [
        "crossbody bag", "wallet", "beanie", "cap", "sunglasses", "belt"
    ],
}

brand_prefixes = [
    "North", "Trail", "Urban", "Peak", "Summit", "Eco", "Nimbus", "Crest", "Pioneer", "Vertex",
    "Atlas", "Terra", "Edge", "Stride", "Motive", "Kindred", "Nomad", "Prime", "Orbit", "Harbor"
]
brand_suffixes = ["Co", "Lab", "Pro", "Works", "Studio", "Gear", "Supply", "House", "Line", "Collective"]

materials = [
    "polyester", "nylon", "cotton", "recycled polyester", "merino wool", "mesh",
    "stainless steel", "aluminum", "BPA-free plastic", "glass", "foam", "rubber",
    "TPU", "softshell fabric", "canvas", "microfiber", "ceramic"
]

colors = [
    "Black", "White", "Gray", "Charcoal", "Slate", "Navy", "Midnight Blue", "Royal Blue", "Sky Blue",
    "Teal", "Forest Green", "Olive", "Lime", "Burgundy", "Crimson", "Coral", "Sand", "Beige",
    "Tan", "Ivory", "Lavender", "Plum", "Mustard", "Orange", "Rust", "Rose", "Mint", "Aqua"
]

sizes = ["XS", "S", "M", "L", "XL", "XXL", "One Size", "6", "7", "8", "9", "10", "11", "12"]

use_cases = {
    "Outdoor": ["hiking", "camping", "backpacking", "trail travel", "cold-weather layering"],
    "Clothing": ["daily wear", "commuting", "travel", "layering", "weekend use"],
    "Electronics": ["commuting", "focus time", "travel", "work calls", "daily listening"],
    "Home": ["small spaces", "daily routines", "gift use", "apartment living", "family use"],
    "Fitness": ["home workouts", "studio sessions", "mobility work", "warm-ups", "travel fitness"],
    "Kids": ["school days", "weekend trips", "outdoor play", "lunch packing", "rainy commutes"],
    "Sports": ["training", "match days", "running", "warm-ups", "gym sessions"],
    "Beauty": ["daily skincare", "travel kits", "dry weather", "morning routine", "sensitive skin"],
    "Kitchen": ["meal prep", "everyday cooking", "quick cleanup", "baking", "small kitchens"],
    "Travel": ["air travel", "packing light", "weekend trips", "business travel", "carry-on organization"],
    "Pet": ["walks", "travel", "home comfort", "feeding", "grooming"],
    "Office": ["remote work", "desk setup", "productivity", "study", "organization"],
    "Footwear": ["walking", "training", "travel", "casual wear", "light hiking"],
    "Accessories": ["everyday carry", "travel", "gift use", "street style", "organization"],
}

benefit_phrases = [
    "lightweight feel", "durable construction", "easy care", "breathable design",
    "soft-touch finish", "compact storage", "all-day comfort", "weather resistance",
    "quick-dry performance", "secure closure", "balanced fit", "versatile styling"
]

def make_brand(i):
    return f"{brand_prefixes[i % len(brand_prefixes)]}{brand_suffixes[i % len(brand_suffixes)]}"

def make_category_label(i):
    return f"{departments[i % len(departments)]}-{(i % 999) + 1:03d}"

def product_sentence(dept, name, brand, price):
    templates = [
        f"{brand} {name} built for {py_rng.choice(use_cases[dept])} with {py_rng.choice(benefit_phrases)}.",
        f"A {py_rng.choice(['reliable', 'versatile', 'easy-to-style', 'travel-friendly'])} {name} that works well for {py_rng.choice(use_cases[dept])}.",
        f"Designed for {py_rng.choice(use_cases[dept])}, this {name} combines {py_rng.choice(benefit_phrases)} and a clean everyday look.",
    ]
    detail = py_rng.choice([
        f"Priced at around ${price:.2f}, it offers a practical balance of features and value.",
        f"Expect a comfortable fit, modern styling, and materials chosen for regular use.",
        f"It is suited to shoppers who want dependable performance without overpaying.",
    ])
    return f"{py_rng.choice(templates)} {detail}"

def make_attributes(dept):
    if dept in ["Outdoor", "Clothing", "Kids", "Sports", "Footwear"]:
        season = py_rng.choice(["all-season", "winter-ready", "spring-friendly", "rainy-season", "cool-weather"])
        fit = py_rng.choice(["regular fit", "athletic fit", "relaxed fit", "tailored fit"])
        return {
            "season": season,
            "fit": fit,
            "material": py_rng.choice(materials),
            "features": "; ".join(py_rng.sample([
                "water-resistant", "breathable", "lightweight", "stretch fabric", "quick-dry",
                "reflective details", "reinforced stitching", "zip pockets", "adjustable cuffs", "machine washable"
            ], 3))
        }
    if dept in ["Electronics"]:
        return {
            "battery_life": f"{py_rng.randint(6, 40)} hours",
            "connectivity": py_rng.choice(["Bluetooth 5.2", "Bluetooth 5.3", "USB-C", "Wi-Fi 6", "NFC"]),
            "material": py_rng.choice(materials),
            "features": "; ".join(py_rng.sample([
                "fast pairing", "noise isolation", "compact charging case", "voice assistant support",
                "low-latency mode", "multi-device connection", "touch controls", "portable design"
            ], 3))
        }
    if dept in ["Home", "Kitchen"]:
        return {
            "capacity": f"{py_rng.choice([0.5, 1, 1.2, 1.5, 2, 3, 5])}L",
            "material": py_rng.choice(materials),
            "features": "; ".join(py_rng.sample([
                "easy-clean design", "space-saving footprint", "dishwasher safe parts", "quiet operation",
                "non-slip base", "temperature control", "energy efficient", "compact storage"
            ], 3))
        }
    if dept in ["Beauty"]:
        return {
            "skin_type": py_rng.choice(["normal", "dry", "oily", "combination", "sensitive"]),
            "material": py_rng.choice(materials),
            "features": "; ".join(py_rng.sample([
                "fragrance-free", "non-greasy", "fast absorbing", "hydrating", "dermatologist tested",
                "travel-friendly", "daily use", "lightweight texture"
            ], 3))
        }
    if dept in ["Travel", "Office", "Accessories", "Pet"]:
        return {
            "material": py_rng.choice(materials),
            "features": "; ".join(py_rng.sample([
                "organized compartments", "durable zippers", "compact form factor", "easy carry",
                "lightweight build", "versatile design", "giftable packaging", "daily convenience"
            ], 3))
        }
    return {
        "material": py_rng.choice(materials),
        "features": "; ".join(py_rng.sample([
            "comfortable to use", "easy maintenance", "versatile styling", "durable build",
            "practical for everyday use", "lightweight design", "value-focused", "premium feel"
        ], 3))
    }

def make_title(dept, name, brand):
    modifiers = ["Pro", "Lite", "Everyday", "Trail", "Active", "Comfort", "Performance", "Travel", "Core", "Max"]
    return f"{brand} {name} {py_rng.choice(modifiers)}"

def make_products(n):
    rows = []
    for i in range(n):
        dept = departments[i % len(departments)]
        name = py_rng.choice(product_templates[dept])
        brand = make_brand(i)
        category = make_category_label(i)
        base_price = round(float(rng.uniform(8, 350)), 2)
        attrs = make_attributes(dept)
        age_restricted = bool(rng.random() < 0.02)
        min_age = 18 if age_restricted else 0
        rows.append({
            "product_id": f"P{i+1:09d}",
            "title": make_title(dept, name, brand),
            "description": product_sentence(dept, name, brand, base_price),
            "department": dept,
            "category": category,
            "brand": brand,
            "base_price": base_price,
            "age_restricted": age_restricted,
            "min_age": min_age,
            "rating": round(float(rng.uniform(3.0, 5.0)), 1),
            "active": True,
            **attrs
        })
    return pd.DataFrame(rows)

def make_variants(products_df):
    n = COUNTS["product_variants"]
    rows = []
    for i in range(n):
        p = products_df.iloc[i % len(products_df)]
        rows.append({
            "variant_id": f"V{i+1:09d}",
            "product_id": p["product_id"],
            "color": colors[i % len(colors)],
            "size": sizes[i % len(sizes)],
            "sku": f"SKU-{i+1:010d}",
        })
    return pd.DataFrame(rows)

def make_customers(n):
    rows = []
    for i in range(n):
        rows.append({
            "customer_id": f"C{i+1:08d}",
            "full_name": f"Customer {i+1:08d}",
            "email": f"customer{i+1:08d}@example.com",
            "created_at": (datetime(2024, 1, 1) + timedelta(days=int(i % 365))).strftime("%Y-%m-%d"),
        })
    return pd.DataFrame(rows)

def make_preferences(customers_df):
    rows = []
    for i, row in customers_df.iterrows():
        dept = departments[i % len(departments)]
        rows.append({
            "customer_id": row["customer_id"],
            "preferred_category": dept,
            "budget_max": round(float(rng.uniform(25, 250)), 2),
            "style_notes": py_rng.choice([
                "prefers minimal everyday pieces",
                "likes outdoor gear and practical details",
                "wants premium-looking basics",
                "shops for value and durability",
                "prefers lightweight travel-friendly items",
                "likes bright colors and sporty fits"
            ]),
            "updated_at": (datetime(2025, 1, 1) + timedelta(days=int(i % 180))).strftime("%Y-%m-%d"),
        })
    return pd.DataFrame(rows)

def make_orders(customers_df):
    n = COUNTS["orders"]
    rows = []
    start = datetime(2025, 1, 1)
    for i in range(n):
        cid = customers_df.iloc[i % len(customers_df)]["customer_id"]
        order_date = start + timedelta(days=int(i % 365), hours=int(i % 24))
        rows.append({
            "order_id": f"O{i+1:09d}",
            "customer_id": cid,
            "order_date": order_date.strftime("%Y-%m-%d %H:%M:%S"),
            "status": py_rng.choice(["pending", "confirmed", "packed", "in_transit", "delivered", "returned"]),
            "shipping_method": py_rng.choice(["standard", "expedited", "same_day"]),
            "total_amount": round(float(rng.uniform(20, 600)), 2),
        })
    return pd.DataFrame(rows)

def make_order_items(orders_df, variants_df):
    n = COUNTS["order_items"]
    rows = []
    for i in range(n):
        oid = orders_df.iloc[i % len(orders_df)]["order_id"]
        vid = variants_df.iloc[i % len(variants_df)]["variant_id"]
        qty = int(rng.integers(1, 4))
        unit_price = round(float(rng.uniform(10, 300)), 2)
        rows.append({
            "order_item_id": f"OI{i+1:010d}",
            "order_id": oid,
            "variant_id": vid,
            "quantity": qty,
            "unit_price": unit_price,
            "line_total": round(qty * unit_price, 2),
        })
    return pd.DataFrame(rows)

def make_tracking(orders_df):
    n = COUNTS["order_tracking"]
    rows = []
    for i in range(n):
        oid = orders_df.iloc[i % len(orders_df)]["order_id"]
        status = py_rng.choice(["pending", "packed", "in_transit", "out_for_delivery", "delivered", "returned"])
        rows.append({
            "tracking_id": f"T{i+1:010d}",
            "order_id": oid,
            "carrier": py_rng.choice(["UPS", "FedEx", "USPS", "DHL", "BlueDart"]),
            "tracking_status": status,
            "eta_date": (datetime(2025, 1, 1) + timedelta(days=int(i % 20))).strftime("%Y-%m-%d"),
            "last_event_at": (datetime(2025, 1, 1) + timedelta(days=int(i % 20), hours=int(i % 24))).strftime("%Y-%m-%d %H:%M:%S"),
        })
    return pd.DataFrame(rows)

def review_text(product_row, rating):
    dept = product_row["department"]
    name = product_row["title"]
    positives = py_rng.sample([
        "comfortable fit", "good value", "nice finishing", "easy to use", "solid build",
        "well-sized pockets", "clean design", "lightweight feel", "useful features", "reliable performance"
    ], 3)
    negatives = py_rng.sample([
        "the packaging was plain", "delivery took an extra day", "the color looked slightly different in person",
        "it needed a short break-in period", "the material felt thinner than expected",
        "the sizing runs a touch narrow", "I wish it had one more pocket", "it is not the cheapest option"
    ], 2)

    if rating >= 5:
        lead = py_rng.choice([
            f"{name} exceeded my expectations.",
            f"I was genuinely impressed by {name}.",
            f"{name} ended up being exactly what I needed."
        ])
        body = f"{lead} It has {positives[0]} and {positives[1]}, and it worked well for {py_rng.choice(use_cases[dept])}. This was exactly what I was hoping for. I would buy it again."
    elif rating >= 4:
        lead = py_rng.choice([
            f"{name} is a strong choice for the price.",
            f"I am happy with {name} overall.",
            f"{name} performs well in day-to-day use."
        ])
        body = f"{lead} The main positives were {positives[0]}, {positives[1]}, and {positives[2]}. {negatives[0].capitalize()} but it was still worth it."
    elif rating >= 3:
        body = (
            f"{name} is acceptable, but not perfect. It has {positives[0]} and {positives[1]}, "
            f"yet {negatives[0]}. For {py_rng.choice(use_cases[dept])}, it should work, but I would compare alternatives first."
        )
    elif rating >= 2:
        body = (
            f"{name} had a few promising details, including {positives[0]}, but {negatives[0]} and {negatives[1]}. "
            f"I expected better quality for the category."
        )
    else:
        body = (
            f"{name} did not meet expectations. The main issue was {negatives[0]}, and {negatives[1]}. "
            f"I would not recommend it unless the buyer is looking for a very basic option."
        )
    return body

def make_reviews(orders_df, products_df):
    n = COUNTS["reviews"]
    rows = []
    for i in range(n):
        order = orders_df.iloc[i % len(orders_df)]
        product = products_df.iloc[i % len(products_df)]
        rating = int(rng.choice([1, 2, 3, 4, 5], p=[0.05, 0.10, 0.20, 0.35, 0.30]))
        rows.append({
            "review_id": f"R{i+1:010d}",
            "order_id": order["order_id"],
            "product_id": product["product_id"],
            "rating": rating,
            "review_title": py_rng.choice([
                "Great value", "Good but not perfect", "Works well", "Very impressed",
                "Could be better", "Solid everyday choice", "Exactly as expected"
            ]),
            "review_text": review_text(product, rating),
            "review_date": (datetime(2025, 1, 1) + timedelta(days=int(i % 365))).strftime("%Y-%m-%d"),
        })
    return pd.DataFrame(rows)

def make_inventory(variants_df):
    n = COUNTS["inventory_snapshots"]
    rows = []
    for i in range(n):
        vid = variants_df.iloc[i % len(variants_df)]["variant_id"]
        stock_qty = int(rng.integers(0, 500))
        reserved_qty = int(rng.integers(0, min(50, stock_qty + 1)))
        available_qty = max(0, stock_qty - reserved_qty)
        rows.append({
            "inventory_snapshot_id": f"I{i+1:010d}",
            "variant_id": vid,
            "warehouse_id": f"WH{(i % 35) + 1:03d}",
            "stock_qty": stock_qty,
            "reserved_qty": reserved_qty,
            "available_qty": available_qty,
            "snapshot_at": (datetime(2025, 1, 1) + timedelta(days=int(i % 30))).strftime("%Y-%m-%d %H:%M:%S"),
        })
    return pd.DataFrame(rows)

def make_behavior(customers_df, products_df):
    n = COUNTS["behavior_events"]
    rows = []
    queries = [
        "waterproof jacket under 80 for hiking", "cheap running shoes", "best backpack for travel",
        "green rain jacket", "comfortable workout shorts", "gift for 10 year old",
        "wireless earbuds under 100", "office desk organizer", "coffee maker for small kitchen"
    ]
    for i in range(n):
        cid = customers_df.iloc[i % len(customers_df)]["customer_id"]
        pid = products_df.iloc[i % len(products_df)]["product_id"]
        rows.append({
            "event_id": f"E{i+1:010d}",
            "customer_id": cid,
            "session_id": f"S{i % 500000:09d}",
            "event_type": py_rng.choice(["view", "search", "click", "add_to_cart", "purchase", "wishlist"]),
            "query_text": py_rng.choice(queries),
            "product_id": pid,
            "event_ts": (datetime(2025, 1, 1) + timedelta(minutes=int(i % 1000000))).strftime("%Y-%m-%d %H:%M:%S"),
        })
    return pd.DataFrame(rows)

def make_logs():
    n = COUNTS["tool_call_logs"]
    rows = []
    tool_names = ["inventory", "tracking", "memory", "catalog", "cart", "pricing"]
    error_types = ["", "", "", "", "timeout", "rate_limit", "connection_error"]
    for i in range(n):
        error = py_rng.choice(error_types)
        success = error == ""
        rows.append({
            "call_id": f"L{i+1:010d}",
            "tool_name": py_rng.choice(tool_names),
            "success": success,
            "latency_ms": int(rng.integers(15, 1200)),
            "error_type": error,
            "ts": (datetime(2025, 1, 1) + timedelta(seconds=int(i % 1000000))).strftime("%Y-%m-%d %H:%M:%S"),
        })
    return pd.DataFrame(rows)

def write_csv(df, name):
    path = OUT_DIR / f"{name}.csv"
    df.to_csv(path, index=False)
    return path

# --------------------
# Generate tables
# --------------------
products_df = make_products(COUNTS["products"])
variants_df = make_variants(products_df)
customers_df = make_customers(COUNTS["customers"])
prefs_df = make_preferences(customers_df)
orders_df = make_orders(customers_df)
items_df = make_order_items(orders_df, variants_df)
tracking_df = make_tracking(orders_df)
reviews_df = make_reviews(orders_df, products_df)
inventory_df = make_inventory(variants_df)
behavior_df = make_behavior(customers_df, products_df)
logs_df = make_logs()

brands_df = pd.DataFrame({"brand": [make_brand(i) for i in range(COUNTS["brands"])]})
categories_df = pd.DataFrame({"category": [make_category_label(i) for i in range(COUNTS["categories"])]})
warehouses_df = pd.DataFrame({
    "warehouse_id": [f"WH{i+1:03d}" for i in range(COUNTS["warehouses"])],
    "warehouse_name": [f"Warehouse {i+1:03d}" for i in range(COUNTS["warehouses"])],
})

# --------------------
# Save
# --------------------
paths = {
    "brands": write_csv(brands_df, "brands"),
    "categories": write_csv(categories_df, "categories"),
    "warehouses": write_csv(warehouses_df, "warehouses"),
    "products": write_csv(products_df, "products"),
    "product_variants": write_csv(variants_df, "product_variants"),
    "customers": write_csv(customers_df, "customers"),
    "customer_preferences": write_csv(prefs_df, "customer_preferences"),
    "orders": write_csv(orders_df, "orders"),
    "order_items": write_csv(items_df, "order_items"),
    "order_tracking": write_csv(tracking_df, "order_tracking"),
    "reviews": write_csv(reviews_df, "reviews"),
    "inventory_snapshots": write_csv(inventory_df, "inventory_snapshots"),
    "behavior_events": write_csv(behavior_df, "behavior_events"),
    "tool_call_logs": write_csv(logs_df, "tool_call_logs"),
}

with open(OUT_DIR / "counts.json", "w") as f:
    json.dump(COUNTS, f, indent=2)

print("Output folder:", OUT_DIR)
print("Counts:", COUNTS)
for k, v in paths.items():
    print(k, "->", v)


In [ ]:

# Quick validation
for name in ["products", "reviews", "inventory_snapshots", "behavior_events"]:
    df = pd.read_csv(OUT_DIR / f"{name}.csv", nrows=3)
    print(f"\n{name}")
    display(df)
